<h1>📄 Biofilter — Report notebook template</h1>

Copy this file to `reports__<report_name>.ipynb` and fill it in. Every
report gets one, and it ships with the report — the code lives in
`biofilter/modules/report/reports/report_<name>.py`, the reference in
`reports_explain/report_<name>.md`, and the worked example here.

Keep the section numbering: people move between these notebooks and the
sections should mean the same thing in each.

Delete this cell when you copy.

<h1>🧬 Biofilter — Report: <code>&lt;report_name&gt;</code></h1>

One paragraph: what question this answers, for whom, and what one row of
the output represents.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = "/path/to/biofilter_data/bundles/20260914"
REPORT = "<report_name>"

bf = Biofilter(bundle=BUNDLE, debug_mode=False)

# Results land here whatever directory the kernel was started in — VS Code
# and Jupyter disagree about that, and a bare filename ends up wherever
# they landed. The project root is the folder holding .biofilter.toml.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Show the smallest input that demonstrates the report, and include one
input you expect to **fail** to resolve — how a report reports absence
is part of what a reader needs to know.

In [ ]:
result = bf.report.run(REPORT, input_data=[...])

df = result.to_pandas()
print(f"{result.num_rows} rows from bundle {result.provenance['bundle_id']}")
df.head()

### 4. Reading the result

Explain the columns a reader could misread. Two that come up in every
report:

- **What `status` values mean**, and whether unresolved inputs are kept.
- **Where null differs from zero.** Null usually means "not computed" or
  "not applicable"; zero means "computed, and none". Say which is which.

If the report has list-valued columns, note that CSV writes them as JSON
while parquet and DataFrames keep them as lists.

In [ ]:
df[["...", "status", "note"]]

### 5. Parameters worth knowing

One cell per parameter that changes the answer rather than the shape —
especially any that trade completeness for speed.

In [ ]:
alternative = bf.report.run(REPORT, input_data=[...], some_param=False)
alternative.to_pandas().head()

### 6. At scale

If the report supports `__ALL__` or accepts large inputs, show it with a
timing, and say what the expensive part is.

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__")
print(f"{everything.num_rows:,} rows in {time.perf_counter() - started:.1f}s")

### 7. If your report needs more than one table

A report returns one table. When its answer is genuinely two shapes —
the rows and what was rejected; the bins and what went into them — call
`self.emit("<name>", table)` in `run()` instead of flattening them
together or writing the second one out as a file. A file is something
the result only names, which is how it stops being checked.

When the report copes with something the reader should know about, call
`self.warn(message, **context)`. It logs at WARNING *and* records the
same thing in `provenance["warnings"]`, which is what reaches whoever
opens the result months later without the log.

```python
def run(self):
    ...
    if dropped:
        self.warn(f"{dropped:,} rows had no coordinates.", rows=dropped)
        self.emit("dropped", dropped_table)
    return main_table
```

In [ ]:
# What this run produced, beyond the main table.
print("tables :", {n: t.num_rows for n, t in result.tables.items()})
print("warnings:", result.provenance["warnings"])
print("files  :", [a.name for a in result.artifacts])

### 8. Export

Always show both, and say what the provenance sidecar is for: ids in a
result are scoped to the bundle that produced them.

In [ ]:
for path in result.write(OUTPUT_DIR / "<report_name>.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter --bundle /path/to/bundles/20260914 report run \
    --report-name <report_name> \
    --input ... \
    --output out.csv
```

### 10. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))